### Source Tables:
- _exponent._bronze_allscripts_tw_works_vw.dbo_patient_member
- _exponent._bronze_allscripts_tw_works_vw.dbo_person
- _exponent._bronze_allscripts_tw_works_vw.dbo_person_other
- _exponent._bronze_allscripts_tw_works_vw.dbo_ethnicity_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_race_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de

### To Do:
- dbo_person_address
  - map this to OMOP.LOCATION once the it has been created
- Add merge logic to gold to look for changes and not blanket overwrite
- Create rolling window for updates
### Notes:
- CARE_SITE, LOCATION, and PROVIDER will all need to run before PERSON
- Should source_to_person and write to gold happen in their own notebooks?  These can run independent of sources 

# Transformation

In [0]:
source = 'allscripts_tw'


In [0]:
silver_person_df = spark.sql(f'''
SELECT 
  -- CONCAT('{source}', ' | ', dbo_person.id) AS person_id, -- will be created in source_to_person/gold
  COALESCE(gender_concept.omop_concept_id, 0) AS gender_concept_id,
  YEAR(dbo_person.dateofbirth) AS year_of_birth,
  MONTH(dbo_person.dateofbirth) AS month_of_birth,
  DAY(dbo_person.dateofbirth) AS day_of_birth,
  dbo_person.dateofbirth AS birth_datetime,
  COALESCE(race_concept.omop_concept_id, 0) AS race_concept_id,
  COALESCE(ethnicity_concept.omop_concept_id, 0) AS ethnicity_concept_id,
  NULL AS location_id,
  NULL AS provider_id,
  NULL care_site_id,
  CONCAT('{source}', ' | ', dbo_person.id) AS person_source_value,
  dbo_sex_de.entryname AS gender_source_value,
  0 AS gender_source_concept_id,
  dbo_race_de.entryname AS race_source_value,
  0 AS race_source_concept_id,
  dbo_ethnicity_de.entryname AS ethnicity_source_value,
  0 AS ethnicity_source_concept_id,
  '{source}' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person_other
ON dbo_person_other.id = dbo_person.id
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_sex_de
ON dbo_sex_de.id = dbo_person.sexde
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_race_de
ON dbo_race_de.id = dbo_person_other.racede
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ethnicity_de
ON dbo_ethnicity_de.id = dbo_person_other.ethnicityde
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept gender_concept
ON gender_concept.source_id = dbo_person.sexde
AND gender_concept.domain_id = 'Gender'
AND gender_concept.source_system = '{source}'
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept race_concept
ON race_concept.source_id = dbo_person_other.racede
AND race_concept.domain_id = 'Race'
AND race_concept.source_system = '{source}'
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept ethnicity_concept
ON ethnicity_concept.source_id = dbo_person_other.ethnicityde
AND ethnicity_concept.domain_id = 'Ethnicity'
AND ethnicity_concept.source_system = '{source}'

WHERE dbo_person.isinactiveflag = 'N'
AND dbo_person.etl_load_ts BETWEEN  CURRENT_DATE() - INTERVAL 14 DAY AND CURRENT_DATE()
''')

display(silver_person_df)
silver_person_df.createOrReplaceTempView("silver_person")


In [0]:
%sql
MERGE INTO _exponent.omop_silver.person AS t
USING silver_person AS s
ON t.person_source_value = s.person_source_value

WHEN MATCHED AND (
     NOT (t.gender_concept_id <=> s.gender_concept_id)
  OR NOT (t.year_of_birth <=> s.year_of_birth)
  OR NOT (t.month_of_birth <=> s.month_of_birth)
  OR NOT (t.day_of_birth <=> s.day_of_birth)
  OR NOT (t.birth_datetime <=> s.birth_datetime)
  OR NOT (t.race_concept_id <=> s.race_concept_id)
  OR NOT (t.ethnicity_concept_id <=> s.ethnicity_concept_id)
  OR NOT (t.location_id <=> s.location_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.care_site_id <=> s.care_site_id)
  -- OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.gender_source_value <=> s.gender_source_value)
  OR NOT (t.gender_source_concept_id <=> s.gender_source_concept_id)
  OR NOT (t.race_source_value <=> s.race_source_value)
  OR NOT (t.race_source_concept_id <=> s.race_source_concept_id)
  OR NOT (t.ethnicity_source_value <=> s.ethnicity_source_value)
  OR NOT (t.ethnicity_source_concept_id <=> s.ethnicity_source_concept_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.gender_concept_id           = s.gender_concept_id,
  t.year_of_birth               = s.year_of_birth,
  t.month_of_birth              = s.month_of_birth,
  t.day_of_birth                = s.day_of_birth,
  t.birth_datetime              = s.birth_datetime,
  t.race_concept_id             = s.race_concept_id,
  t.ethnicity_concept_id        = s.ethnicity_concept_id,
  t.location_id                 = s.location_id,
  t.provider_id                 = s.provider_id,
  t.care_site_id                = s.care_site_id,
  -- t.person_source_value         = s.person_source_value,
  t.gender_source_value         = s.gender_source_value,
  t.gender_source_concept_id    = s.gender_source_concept_id,
  t.race_source_value           = s.race_source_value,
  t.race_source_concept_id      = s.race_source_concept_id,
  t.ethnicity_source_value      = s.ethnicity_source_value,
  t.ethnicity_source_concept_id = s.ethnicity_source_concept_id,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  -- person_id,
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  birth_datetime,
  race_concept_id,
  ethnicity_concept_id,
  location_id,
  provider_id,
  care_site_id,
  person_source_value,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.gender_concept_id,
  s.year_of_birth,
  s.month_of_birth,
  s.day_of_birth,
  s.birth_datetime,
  s.race_concept_id,
  s.ethnicity_concept_id,
  s.location_id,
  s.provider_id,
  s.care_site_id,
  s.person_source_value,
  s.gender_source_value,
  s.gender_source_concept_id,
  s.race_source_value,
  s.race_source_concept_id,
  s.ethnicity_source_value,
  s.ethnicity_source_concept_id,
  s.source_system,
  current_timestamp()
);


In [0]:
%sql
SELECT * FROM _exponent.omop_silver.person

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_person (
    source_system,
    person_source_value,
    -- person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.person_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT source_system, person_source_value, last_mod_tsp
    FROM _exponent.omop_silver.person
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_person x
  ON s.person_source_value = x.person_source_value;

In [0]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_person 

In [0]:
%sql
MERGE INTO _exponent.omop.person AS gold_person
USING (
  SELECT
    source_to_person.person_id,                    
    s.gender_concept_id,
    s.year_of_birth,
    s.month_of_birth,
    s.day_of_birth,
    s.birth_datetime,
    s.race_concept_id,
    s.ethnicity_concept_id,
    s.location_id,
    s.provider_id,
    s.care_site_id,
    s.person_source_value,
    s.gender_source_value,
    s.gender_source_concept_id,
    s.race_source_value,
    s.race_source_concept_id,
    s.ethnicity_source_value,
    s.ethnicity_source_concept_id
  FROM _exponent.omop_silver.person s
  JOIN _exponent.omop_mapping.source_to_person
    ON source_to_person.person_source_value = s.person_source_value
   AND source_to_person.active_flag = TRUE
) AS src
ON gold_person.person_id = src.person_id

WHEN MATCHED THEN UPDATE SET
  gold_person.gender_concept_id = src.gender_concept_id,
  gold_person.year_of_birth = src.year_of_birth,
  gold_person.month_of_birth = src.month_of_birth,
  gold_person.day_of_birth = src.day_of_birth,
  gold_person.birth_datetime = src.birth_datetime,
  gold_person.race_concept_id = src.race_concept_id,
  gold_person.ethnicity_concept_id = src.ethnicity_concept_id,
  gold_person.location_id = src.location_id,
  gold_person.provider_id = src.provider_id,
  gold_person.care_site_id = src.care_site_id,
  gold_person.person_source_value = src.person_source_value,
  gold_person.gender_source_value = src.gender_source_value,
  gold_person.gender_source_concept_id = src.gender_source_concept_id,
  gold_person.race_source_value = src.race_source_value,
  gold_person.race_source_concept_id = src.race_source_concept_id,
  gold_person.ethnicity_source_value = src.ethnicity_source_value,
  gold_person.ethnicity_source_concept_id = src.ethnicity_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  birth_datetime,
  race_concept_id,
  ethnicity_concept_id,
  location_id,
  provider_id,
  care_site_id,
  person_source_value,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id
)
VALUES (
  src.person_id,
  src.gender_concept_id,
  src.year_of_birth,
  src.month_of_birth,
  src.day_of_birth,
  src.birth_datetime,
  src.race_concept_id,
  src.ethnicity_concept_id,
  src.location_id,
  src.provider_id,
  src.care_site_id,
  src.person_source_value,
  src.gender_source_value,
  src.gender_source_concept_id,
  src.race_source_value,
  src.race_source_concept_id,
  src.ethnicity_source_value,
  src.ethnicity_source_concept_id
);


In [0]:
# %sql
# SELECT * FROM _exponent.omop.person